# LC 191 — Number of 1 Bits
**Difficulty:** Easy | **Category:** Bit Manipulation
**Pattern:** Brian Kernighan's Bit Trick

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Every time you do <code>n & (n-1)</code>,
you erase the lowest set bit. Count how many times you can do
this before n reaches zero — that count is your answer.
</div>

## Official Problem Statement

Write a function that takes the binary representation of a positive
integer and returns the number of set bits it has
(also known as the Hamming weight).

**Constraints:**
- `1 <= n <= 2^31 - 1`
- Input is treated as an **unsigned** 32-bit integer

## What This Is Actually Asking

You get a number. Write it in binary. Count the 1s.

That's it. The number 11 in binary is 1011, which has three 1s.

The trick is doing this efficiently without converting to a string.

Each 1-bit means a power of 2 is "on" in the number.

## Walk Through an Example by Hand

Input: `n = 11`  →  binary: `1011`

**Step 1:** `n = 1011`, `n-1 = 1010`
```
  1011
& 1010
------
  1010   ← lowest set bit (bit 0) erased, count = 1
```
**Step 2:** `n = 1010`, `n-1 = 1001`
```
  1010
& 1001
------
  1000   ← bit 1 erased, count = 2
```
**Step 3:** `n = 1000`, `n-1 = 0111`
```
  1000
& 0111
------
  0000   ← bit 3 erased, count = 3
```
n == 0, stop. **Answer: 3**

## The Picture

```
n = 11 (decimal)

Bit position:  7  6  5  4  3  2  1  0
Binary:        0  0  0  0  1  0  1  1
                            ^     ^ ^
                            |     | +-- bit 0 is ON
                            |     +---- bit 1 is ON
                            +---------- bit 3 is ON
Total 1-bits: 3

Brian Kernighan trick:
  n & (n-1)  always clears the RIGHTMOST 1-bit

  n    = ...X 1 0 0 0   (rightmost 1 followed by zeros)
  n-1  = ...X 0 1 1 1   (borrows all zeros, flips the 1)
  AND  = ...X 0 0 0 0   (rightmost 1 gone!)

Each iteration removes exactly one 1-bit.
Number of iterations == number of 1-bits.
```

## When To Use This Pattern

- When you need to **count set bits** in a number.
- When you see **Hamming weight** or **popcount** in a problem.
- When you need to check if a number is a **power of 2**
  (power of 2 has exactly one 1-bit → `n & (n-1) == 0`).
- When iterating bit-by-bit would be slow and
  you want O(number of set bits) instead of O(32).

## The Approach

Start a counter at zero. While n is not zero, apply `n = n & (n-1)`
and increment the counter. Each application removes exactly one
set bit. When n reaches zero, all set bits have been counted.
Return the counter.

In [1]:
from typing import List  # standard collection types

In [2]:
def test_harness(func):
    tests = [
        # (input, expected, label)
        (11,         3, "11 = 1011, three 1s"),
        (128,        1, "128 = 10000000, one 1"),
        (2147483645, 30, "2^31-3, thirty 1s"),
        (1,          1, "edge: single bit"),
        (2147483647, 31, "2^31-1, all 31 bits on"),
    ]
    passed = 0
    for n, expected, label in tests:
        result = func(n)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"  [{status}] {label}")
        if status == "FAILED":
            print(f"           got={result}, expected={expected}")
    print(f"\n  {passed}/{len(tests)} tests passed")

In [4]:
def hammingWeight(n: int) -> int:
    
    """
    Count the number of 1 bits in integer n.

    Approach: Brian Kernighan's trick.
      n & (n-1) clears the lowest set bit.
      Repeat until n == 0. Count iterations.

    Time:  O(k) where k = number of set bits (at most 31)
    Space: O(1)
    """
    res = 0 
    while n:
        n &= (n-1)
        res += 1
    return res



# Debug prints — expected values shown in comments
print(hammingWeight(11))          # expected: 3
print(hammingWeight(128))         # expected: 1
print(hammingWeight(2147483645))  # expected: 30
print(hammingWeight(1))           # expected: 1
print(hammingWeight(2147483647))  # expected: 31
test_harness (hammingWeight)

3
1
30
1
31
  [PASSED] 11 = 1011, three 1s
  [PASSED] 128 = 10000000, one 1
  [PASSED] 2^31-3, thirty 1s
  [PASSED] edge: single bit
  [PASSED] 2^31-1, all 31 bits on

  5/5 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(hammingWeight)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force (check each of 32 bits) | O(32) = O(1) | O(1) |
| Brian Kernighan (clear lowest set bit) | O(k), k = set bits | O(1) |
| Python built-in `bin(n).count('1')` | O(32) = O(1) | O(1) |

## Real World Connection

At Citi, each of the 6,000 monitored endpoints can carry a bitmask
representing which health checks are currently failing.
Counting set bits tells an on-call engineer how many checks are
failing at a glance without iterating through every flag.
In AWS serverless telemetry, Lambda function permission sets are
often stored as bitmasks; a popcount reveals how many permissions
are active, useful for audit and compliance ETL pipelines.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra